# DevSecOps Agent Team — אבן דרך 1

רן בן חמו, חן קדוש

המחברת רק מריצה את הקוד שבריפו (קבצי `.py`). כל הקוד נמצא בריפו, לא בתאים.
בלי מפתח היא רצה עם המודל המדומה (`scripted`), בלי רשת למודל ובלי עלות. אפשר להריץ "Run all" מלמעלה למטה.

## 1. הורדת הריפו

משכפלים את הריפו הציבורי לתיקייה `/content/devsecops-team` ונכנסים אליה.
`REF` הוא הענף, התג או ה־commit שבודקים. כדי לבדוק את הגרסה שהוגשה, שנו אותו לתג או ל־commit שהוגש.

In [ ]:
import os

REPO_URL = "https://github.com/RanBenHemo/devsecops-team.git"
REF = "main"
TARGET = "/content/devsecops-team"

if not os.path.isdir(TARGET):
    !git clone --quiet {REPO_URL} {TARGET}
%cd {TARGET}
!git checkout --quiet {REF}
!git log --oneline -1

## 2. התקנת התלויות

כל התלויות ב־`requirements.txt` נעולות לגרסה מדויקת.

In [ ]:
%pip install -q -r requirements.txt

## 3. הרצת הטסטים

פקודה אחת, בלי מפתח, בלי רשת ובלי עלות. השורה האחרונה היא הסיכום (כמה טסטים עברו).

In [ ]:
!python -m pytest -q

## 4. דמו עם המודל המדומה (בלי מפתח)

`main.py review samples/login.py` מריץ את שלושת הסוכנים על קובץ הדוגמה, וכותב כרטיסי החלטה ל־`reports/`.
התא מדפיס את הדוח ואת תחילת ה־trace (כל צעד של כל סוכן).
קוד יציאה 1 כאן צפוי: הוא אומר שנמצאו חולשות אמיתיות. אף סוכן לא כותב לקובץ המקורי.

In [ ]:
import json
import subprocess
import sys

run = subprocess.run([sys.executable, "main.py", "review", "samples/login.py", "--trace"],
                     capture_output=True, encoding="utf-8")
print("exit code:", run.returncode, "(0 = no real findings, 1 = real findings, 2 = error)")
result = json.loads(run.stdout)
print("report files:", result["report_files"])
print()
print(open(result["report_files"]["md"], encoding="utf-8").read())
steps = [line for line in run.stderr.splitlines() if line.startswith("[trace]")]
print(f"--- trace: the first 30 of {len(steps)} lines ---")
print("\n".join(steps[:30]))

## 5. דמו עם מודל אמיתי (לא חובה)

התא קורא את המפתחות מ־Colab Secrets (סמל המפתח בצד שמאל): `GROQ_API_KEY`, `GEMINI_API_KEY`, `OPENROUTER_API_KEY`,
ואם רוצים גם `GROQ_MODEL`, `GEMINI_MODEL`, `OPENROUTER_MODEL`. הוא מעתיק ל־`os.environ` רק את מה שקיים, ולעולם לא מדפיס ערך.
כל סוכן משתמש בספק אחר. סוכן שאין לו מפתח עוצר עם `missing_api_key`.
אם אין אף מפתח, התא מדפיס שהדמו דולג, ולא נכשל.

In [ ]:
import json
import os
import subprocess
import sys

NAMES = ["GROQ_API_KEY", "GEMINI_API_KEY", "OPENROUTER_API_KEY", "GROQ_MODEL", "GEMINI_MODEL", "OPENROUTER_MODEL"]
found = []
try:
    from google.colab import userdata
    for name in NAMES:
        try:
            value = userdata.get(name)
        except Exception:  # the secret does not exist, or this notebook has no access to it
            continue
        if value:
            os.environ[name] = value
            found.append(name)
except Exception:
    pass  # not running in Colab

keys = [name for name in found if name.endswith("_API_KEY")]
if not keys:
    print("No API key in Colab Secrets: the real-model demo was skipped.")
else:
    print("Keys found (names only):", ", ".join(keys))
    try:
        run = subprocess.run([sys.executable, "main.py", "review", "samples/login.py", "--model", "llm"],
                             capture_output=True, encoding="utf-8", timeout=700)
        print("exit code:", run.returncode)
        result = json.loads(run.stdout)
        print(open(result["report_files"]["md"], encoding="utf-8").read())
    except Exception as error:
        print("The real-model demo did not finish:", type(error).__name__)